### Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

0.5.3
2.2.2
1.26.4
0.13.2


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(exp):

### Parameters

In [3]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

has_transformation = params["has_transformation"]
print("Has transformation:", has_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

Exp:		 exp8
Data variations: ['none']
Has transformation: False
Threshold corr:	 0.5
Groups id:	 ['SecoAmazonas', 'FrescoSanMartin', 'FrescoCusco', 'SecoSanMartin', 'SecoCusco', 'FrescoAmazonas']
Subgroups id:	 {'SecoAmazonas': ['1', '2'], 'FrescoSanMartin': ['1', '2'], 'FrescoCusco': ['1', '2'], 'SecoSanMartin': ['1', '2'], 'SecoCusco': ['1', '2'], 'FrescoAmazonas': ['1', '2']}
Groups id (no):	 ['Blank', 'QC', 'Std']


### Load dataset

In [4]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

,Average Rt,Average Mz,Metabolite name,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,...,SecoCusco_1.3,SecoCusco_2.1,SecoCusco_2.2,SecoCusco_2.3,FrescoAmazonas_1.1,FrescoAmazonas_1.2,FrescoAmazonas_1.3,FrescoAmazonas_2.1,FrescoAmazonas_2.2,FrescoAmazonas_2.3
0,0.089,165.99460,NaN,1.213131e+06,1.177177e+06,1.566258e+05,1.731480e+06,1.699705e+06,6.280596e+06,1.696391e+06,...,3.338787e+05,1.133971e+06,1.808500e+05,3.342117e+05,4.322203e+06,1.821127e+06,1.770069e+06,1.743055e+06,1.463076e+06,6.428159e+06
1,0.090,158.01272,"[Similar to: 1-benzylhexahydropyrimidine-2,4,6...",1.431372e+04,2.609972e+04,5.571270e+03,4.865791e+04,3.923473e+04,8.001311e+03,3.985529e+04,...,4.652185e+03,3.434351e+04,2.178133e+04,3.596103e+03,5.631005e+04,4.148497e+04,9.098716e+03,2.515887e+04,5.974290e+04,5.789815e+03
2,0.097,172.95681,NaN,3.406202e+06,8.569343e+06,2.299997e+06,1.344103e+07,1.831757e+06,1.657309e+06,1.042663e+07,...,2.008602e+06,6.643525e+06,6.447431e+06,2.520397e+06,1.275967e+07,1.578762e+06,4.518729e+06,9.966807e+06,4.552709e+05,1.402781e+07
3,0.191,167.01326,NaN,5.020429e+07,4.827342e+07,1.113485e+06,6.889062e+07,6.861163e+07,2.800838e+07,6.561428e+07,...,2.189862e+06,4.159900e+07,4.101282e+07,1.728290e+06,7.507800e+07,7.443810e+07,3.294308e+07,6.365119e+07,6.403714e+07,2.607451e+07
4,0.196,165.98329,NaN,2.429038e+06,2.564504e+06,1.071655e+06,1.040565e+07,5.333877e+05,1.667311e+06,2.812425e+06,...,1.433978e+06,9.607175e+05,5.082268e+06,2.935405e+06,4.686568e+06,3.386707e+06,7.763971e+05,1.503685e+06,7.711962e+06,2.505585e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,29.965,341.26686,Docosahexaenoic acid,8.376403e+06,8.179919e+06,1.137834e+07,1.124024e+07,1.109478e+07,1.698218e+07,7.006518e+06,...,9.237098e+06,7.589148e+06,6.798410e+06,8.014118e+06,1.125906e+07,7.940634e+06,2.003224e+06,7.171052e+06,8.523936e+06,1.454813e+07
555,29.966,178.15937,N-Propylamphetamine,3.290299e+07,3.307227e+07,6.826276e+07,4.771023e+07,4.564733e+07,7.901293e+07,4.210922e+07,...,5.918501e+07,2.785011e+07,2.457465e+07,1.308641e+07,4.774097e+07,6.472719e+07,1.092961e+08,3.954838e+07,3.810093e+07,8.150158e+07
556,29.969,707.49253,NaN,1.537170e+07,1.921792e+07,4.989073e+06,2.964074e+07,2.545159e+07,5.389525e+06,2.626919e+07,...,2.580318e+06,1.344052e+07,1.132973e+07,1.192044e+06,3.228213e+07,2.365662e+07,3.800210e+06,2.567993e+07,6.028380e+06,1.913270e+06
557,29.969,165.11386,2-piperazinopyrimidine,2.308164e+06,2.275392e+06,6.454852e+06,1.710127e+06,3.178837e+06,8.738433e+06,1.248976e+06,...,5.209212e+06,1.847576e+06,1.788281e+06,1.986319e+06,3.774866e+06,3.512772e+06,1.176852e+07,2.692574e+06,2.835947e+06,7.170966e+06


In [5]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

,Average Rt,Average Mz
0,0.089,165.99460
1,0.090,158.01272
2,0.097,172.95681
3,0.191,167.01326
4,0.196,165.98329
...,...,...
554,29.965,341.26686
555,29.966,178.15937
556,29.969,707.49253
557,29.969,165.11386


In [6]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3,FrescoCusco_1.1,...,SecoCusco_1.3,SecoCusco_2.1,SecoCusco_2.2,SecoCusco_2.3,FrescoAmazonas_1.1,FrescoAmazonas_1.2,FrescoAmazonas_1.3,FrescoAmazonas_2.1,FrescoAmazonas_2.2,FrescoAmazonas_2.3
0,1.213131e+06,1.177177e+06,1.566258e+05,1.731480e+06,1.699705e+06,6.280596e+06,1.696391e+06,1.472273e+06,6.836066e+06,1.086563e+06,...,3.338787e+05,1.133971e+06,1.808500e+05,3.342117e+05,4.322203e+06,1.821127e+06,1.770069e+06,1.743055e+06,1.463076e+06,6.428159e+06
1,1.431372e+04,2.609972e+04,5.571270e+03,4.865791e+04,3.923473e+04,8.001311e+03,3.985529e+04,3.492032e+04,6.318870e+03,3.495953e+04,...,4.652185e+03,3.434351e+04,2.178133e+04,3.596103e+03,5.631005e+04,4.148497e+04,9.098716e+03,2.515887e+04,5.974290e+04,5.789815e+03
2,3.406202e+06,8.569343e+06,2.299997e+06,1.344103e+07,1.831757e+06,1.657309e+06,1.042663e+07,1.012592e+07,8.155710e+06,1.101946e+07,...,2.008602e+06,6.643525e+06,6.447431e+06,2.520397e+06,1.275967e+07,1.578762e+06,4.518729e+06,9.966807e+06,4.552709e+05,1.402781e+07
3,5.020429e+07,4.827342e+07,1.113485e+06,6.889062e+07,6.861163e+07,2.800838e+07,6.561428e+07,6.422012e+07,2.519854e+07,6.552697e+07,...,2.189862e+06,4.159900e+07,4.101282e+07,1.728290e+06,7.507800e+07,7.443810e+07,3.294308e+07,6.365119e+07,6.403714e+07,2.607451e+07
4,2.429038e+06,2.564504e+06,1.071655e+06,1.040565e+07,5.333877e+05,1.667311e+06,2.812425e+06,1.715898e+06,1.772940e+06,1.913464e+06,...,1.433978e+06,9.607175e+05,5.082268e+06,2.935405e+06,4.686568e+06,3.386707e+06,7.763971e+05,1.503685e+06,7.711962e+06,2.505585e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554,8.376403e+06,8.179919e+06,1.137834e+07,1.124024e+07,1.109478e+07,1.698218e+07,7.006518e+06,1.322471e+07,1.424028e+07,1.047806e+07,...,9.237098e+06,7.589148e+06,6.798410e+06,8.014118e+06,1.125906e+07,7.940634e+06,2.003224e+06,7.171052e+06,8.523936e+06,1.454813e+07
555,3.290299e+07,3.307227e+07,6.826276e+07,4.771023e+07,4.564733e+07,7.901293e+07,4.210922e+07,4.038062e+07,9.140048e+07,4.372639e+07,...,5.918501e+07,2.785011e+07,2.457465e+07,1.308641e+07,4.774097e+07,6.472719e+07,1.092961e+08,3.954838e+07,3.810093e+07,8.150158e+07
556,1.537170e+07,1.921792e+07,4.989073e+06,2.964074e+07,2.545159e+07,5.389525e+06,2.626919e+07,2.693046e+07,2.338017e+06,2.590761e+07,...,2.580318e+06,1.344052e+07,1.132973e+07,1.192044e+06,3.228213e+07,2.365662e+07,3.800210e+06,2.567993e+07,6.028380e+06,1.913270e+06
557,2.308164e+06,2.275392e+06,6.454852e+06,1.710127e+06,3.178837e+06,8.738433e+06,1.248976e+06,2.793988e+06,5.279317e+06,2.960870e+06,...,5.209212e+06,1.847576e+06,1.788281e+06,1.986319e+06,3.774866e+06,3.512772e+06,1.176852e+07,2.692574e+06,2.835947e+06,7.170966e+06


In [7]:
df_join_raw_intensity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 559 entries, 0 to 558
Data columns (total 36 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   SecoAmazonas_1.1     559 non-null    float64
 1   SecoAmazonas_1.2     559 non-null    float64
 2   SecoAmazonas_1.3     559 non-null    float64
 3   FrescoSanMartin_1.1  559 non-null    float64
 4   FrescoSanMartin_1.2  559 non-null    float64
 5   FrescoSanMartin_1.3  559 non-null    float64
 6   FrescoSanMartin_2.1  559 non-null    float64
 7   FrescoSanMartin_2.2  559 non-null    float64
 8   FrescoSanMartin_2.3  559 non-null    float64
 9   FrescoCusco_1.1      559 non-null    float64
 10  FrescoCusco_1.2      559 non-null    float64
 11  FrescoCusco_1.3      559 non-null    float64
 12  FrescoCusco_2.1      559 non-null    float64
 13  FrescoCusco_2.2      559 non-null    float64
 14  FrescoCusco_2.3      559 non-null    float64
 15  SecoAmazonas_2.1     559 non-null    float64


In [8]:
check_dataset(df_join_raw_intensity)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 20124


### Generate graphs

In [9]:
""" from sklearn import preprocessing

X_scaled = preprocessing.RobustScaler().fit_transform(df_join_raw)

df_join_raw_log = pd.DataFrame(X_scaled, columns=df_join_raw.columns)
df_join_raw_log """

' from sklearn import preprocessing\n\nX_scaled = preprocessing.RobustScaler().fit_transform(df_join_raw)\n\ndf_join_raw_log = pd.DataFrame(X_scaled, columns=df_join_raw.columns)\ndf_join_raw_log '

In [10]:
# Transformation (log10)

if not has_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3,FrescoSanMartin_1.1,FrescoSanMartin_1.2,FrescoSanMartin_1.3,FrescoSanMartin_2.1,FrescoSanMartin_2.2,FrescoSanMartin_2.3,FrescoCusco_1.1,...,SecoCusco_1.3,SecoCusco_2.1,SecoCusco_2.2,SecoCusco_2.3,FrescoAmazonas_1.1,FrescoAmazonas_1.2,FrescoAmazonas_1.3,FrescoAmazonas_2.1,FrescoAmazonas_2.2,FrescoAmazonas_2.3
0,6.083908,6.070842,5.194863,6.238417,6.230374,6.798001,6.229526,6.167988,6.834806,6.036055,...,5.523589,6.054602,5.257318,5.524022,6.635705,6.260340,6.247990,6.241311,6.165267,6.808087
1,4.155752,4.416636,3.745954,4.687153,4.593671,3.903161,4.600486,4.543078,3.800639,4.543566,...,3.667657,4.535845,4.338084,3.555832,4.750586,4.617891,3.958980,4.400691,4.776286,3.762665
2,6.532270,6.932948,6.361727,7.128433,6.262868,6.219404,7.018144,7.005435,6.911462,7.042160,...,6.302894,6.822399,6.809387,6.401469,7.105840,6.198317,6.655016,6.998556,5.658270,7.146990
3,7.700741,7.683708,6.046684,7.838160,7.836398,7.447288,7.816998,7.807671,7.401375,7.816420,...,6.340417,7.619083,7.612920,6.237617,7.875513,7.871795,7.517764,7.803807,7.806432,7.416216
4,6.385434,6.409003,6.030055,7.017269,5.727043,6.222017,6.449081,6.234491,6.248694,6.281820,...,6.156542,5.982596,6.706058,6.467668,6.670855,6.529778,5.890084,6.177157,6.887165,7.398909


In [11]:
check_dataset(df_join_raw_log)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 20124


In [12]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,SecoAmazonas_1.1,SecoAmazonas_1.2,SecoAmazonas_1.3
0,6.083908,6.070842,5.194863
1,4.155752,4.416636,3.745954
2,6.532270,6.932948,6.361727
3,7.700741,7.683708,6.046684
4,6.385434,6.409003,6.030055


In [13]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 1677


In [14]:
# Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)
dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,0,1,2,3,4,5,6,7,8,9,...,549,550,551,552,553,554,555,556,557,558
0,6.083908,4.155752,6.532270,7.700741,6.385434,8.610974,6.348509,6.538010,6.816724,6.385434,...,6.715411,6.184520,7.322656,6.782134,6.515814,6.923058,7.517235,7.186722,6.363267,7.201043
1,6.070842,4.416636,6.932948,7.683708,6.409003,8.578933,7.777185,5.717278,7.622484,6.409003,...,6.747370,6.169553,7.382571,6.785937,6.465011,6.912749,7.519464,7.283706,6.357056,6.707344
2,5.194863,3.745954,6.361727,6.046684,6.030055,8.524494,6.230455,6.368448,6.953095,6.588367,...,6.376713,6.262426,8.200814,7.791348,6.782885,7.056079,7.834184,6.698020,6.809886,6.353711


In [15]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 0
Count zero:	 0
Count positive:	 1677


In [16]:
# Correlation matrix

dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)
dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

SecoAmazonas 1 (3, 559)
SecoAmazonas 2 (3, 559)
FrescoSanMartin 1 (3, 559)
FrescoSanMartin 2 (3, 559)
FrescoCusco 1 (3, 559)
FrescoCusco 2 (3, 559)
SecoSanMartin 1 (3, 559)
SecoSanMartin 2 (3, 559)
SecoCusco 1 (3, 559)
SecoCusco 2 (3, 559)
FrescoAmazonas 1 (3, 559)
FrescoAmazonas 2 (3, 559)


,0,1,2,3,4,5,6,7,8,9,...,549,550,551,552,553,554,555,556,557,558
0,1.000000,-0.169731,-0.105750,-0.997286,-0.630522,-0.084862,-0.085864,0.068856,-0.071064,0.449032,...,-0.529632,0.345099,0.715816,0.983876,0.347278,0.586296,0.991899,-0.338023,0.906877,-0.022983
1,-0.169731,1.000000,-0.997914,-0.241824,-0.871929,0.967532,-0.996425,0.994838,-0.995061,-0.804336,...,-0.925815,0.983522,-0.566661,0.343252,0.983100,0.897855,0.293539,-0.984855,0.569207,0.981329
2,-0.105750,-0.997914,1.000000,-0.178672,-0.838497,0.981832,-0.999800,0.999314,-0.999394,-0.841020,...,-0.899480,0.969798,-0.618676,0.281895,0.969229,0.867555,0.231207,-0.971607,0.514935,0.991700
3,-0.997286,-0.241824,-0.178672,1.000000,-0.685954,-0.011276,-0.158980,0.142116,-0.144307,0.382032,...,-0.590643,0.413262,0.662464,0.994373,0.415375,0.644346,0.998559,-0.406394,0.935440,0.050681
4,-0.630522,-0.871929,-0.838497,-0.685954,1.000000,0.719864,-0.827443,0.817744,-0.819016,-0.410396,...,-0.992315,0.946081,-0.090654,0.759176,0.946831,0.998447,0.724009,-0.943615,0.898881,0.761475


In [17]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'SecoAmazonas': {'1':           0         1         2         3         4         5         6    \
  0    1.000000 -0.169731 -0.105750 -0.997286 -0.630522 -0.084862 -0.085864   
  1   -0.169731  1.000000 -0.997914 -0.241824 -0.871929  0.967532 -0.996425   
  2   -0.105750 -0.997914  1.000000 -0.178672 -0.838497  0.981832 -0.999800   
  3   -0.997286 -0.241824 -0.178672  1.000000 -0.685954 -0.011276 -0.158980   
  4   -0.630522 -0.871929 -0.838497 -0.685954  1.000000  0.719864 -0.827443   
  ..        ...       ...       ...       ...       ...       ...       ...   
  554  0.586296  0.897855  0.867555  0.644346  0.998447 -0.757420  0.857446   
  555  0.991899  0.293539  0.231207  0.998559  0.724009 -0.042393  0.211725   
  556 -0.338023 -0.984855 -0.971607 -0.406394 -0.943615  0.909057 -0.966686   
  557  0.906877  0.569207  0.514935  0.935440  0.898881 -0.342917  0.497707   
  558 -0.022983  0.981329  0.991700  0.050681  0.761475 -0.998080  0.994070   
  
            7         8     

In [18]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 155986
Count zero:	 0
Count positive:	 156495


In [19]:
# Build graph (corpus graphs)

dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
2,0,3,-0.997286,1
3,0,4,-0.630522,1
9,0,10,-0.989137,1
10,0,11,-0.875317,1
11,0,12,-0.889671,1


In [20]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [21]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

100%|██████████| 6/6 [00:05<00:00,  1.05it/s]


In [22]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

		G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
		list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,SecoAmazonas,1,559,103333,0.662557,NaN,True
1,SecoAmazonas,2,559,104793,0.671918,NaN,True
2,FrescoSanMartin,1,559,104932,0.672809,NaN,True
3,FrescoSanMartin,2,559,105894,0.678977,NaN,True
4,FrescoCusco,1,559,103348,0.662653,NaN,True
5,FrescoCusco,2,559,104638,0.670924,NaN,True
6,SecoSanMartin,1,559,102897,0.659761,NaN,True
7,SecoSanMartin,2,559,102453,0.656914,NaN,True
8,SecoCusco,1,559,102044,0.654292,NaN,True
9,SecoCusco,2,559,102392,0.656523,NaN,True
